# Clinical Trial Intelligence Platform
## Clinical Data Quality Controls

### Objective

This notebook provides an auditable summary of the data-quality controls implemented in the Silver layer across the major clinical entities:

- Subjects
- Visits
- Laboratory Results
- Adverse Events

For each entity, the notebook identifies:

- the implemented DQ rule,
- the type of validation,
- the source attribute being validated,
- the reference or trusted dataset used for validation,
- and the action taken when validation fails.

Invalid records that violate blocking DQ rules are routed to the corresponding quarantine table, while trusted records continue into the Silver layer.

## 1. Data Quality Architecture

The Silver layer acts as the trust boundary of the platform.

Incoming Bronze records are standardized and validated using both record-level business rules and trusted reference/dimension datasets.

**Validation Flow**

Bronze Source  
↓  
Standardization  
↓  
Data Quality + Referential Validation  
↓  
**Valid → Silver**  
**Invalid → Quarantine**

The validation framework includes:

- Completeness checks
- Domain and range checks
- Referential-integrity checks
- Cross-entity relationship checks
- Temporal consistency checks
- Reference-data mapping checks
- Clinical consistency checks


## 2. Subject Data Quality Controls

**Primary source:** `bronze.edc_subjects`

Subject records are validated using the incoming EDC attributes together with trusted study, site, treatment-arm, diagnosis, and sex reference datasets.

**Valid destination:** `silver.subjects`  
**Invalid destination:** `quarantine.subjects`

### Subject DQ Control Matrix

| DQ Rule | Validation Type | Field(s) | Rule / Condition | Reference / Validation Source | Failure Action |
|---|---|---|---|---|---|
| `missing_subject_id` | Completeness | `subject_id` | `subject_id` must not be NULL | Source record | Quarantine |
| `missing_study_id` | Completeness | `study_id` | `study_id` must not be NULL | Source record | Quarantine |
| `unknown_study_id` | Referential Integrity | `study_id` | `study_id` must exist in the study master | `dim_study` | Quarantine |
| `missing_site_id` | Completeness | `site_id` | `site_id` must not be NULL | Source record | Quarantine |
| `unknown_site_id` | Referential Integrity | `site_id` | `site_id` must exist in the site master | `dim_site` | Quarantine |
| `site_not_in_study` | Relationship Integrity | `study_id`, `site_id` | Site must belong to the specified study | `dim_site` | Quarantine |
| `age_out_of_range` | Domain / Range | `age` | **18 ≤ age ≤ 100**; NULL age also fails validation | Business rule | Quarantine |
| `unmappable_sex` | Reference Mapping | `sex` | A non-null source sex value must map to a standardized sex value | `ref_sex` | Quarantine |
| `unknown_baseline_condition` | Reference Mapping | `baseline_condition_code` | A non-null diagnosis code must exist in the diagnosis reference | `ref_diagnosis` | Quarantine |
| `invalid_source_snapshot_date` | Metadata Validation | Source filename | Filename must provide a valid source date in **YYYYMMDD** format | `_source_file_name` | Quarantine |
| `missing_screening_date` | Completeness | `screening_date` | `screening_date` must not be NULL | Source record | Quarantine |
| `enrollment_before_screening` | Temporal Consistency | `screening_date`, `enrollment_date` | `enrollment_date ≥ screening_date` | Source record | Quarantine |
| `consent_after_enrollment` | Temporal Consistency | `informed_consent_date`, `enrollment_date` | `informed_consent_date ≤ enrollment_date` | Source record | Quarantine |
| `randomization_before_enrollment` | Temporal Consistency | `randomization_date`, `enrollment_date` | `randomization_date ≥ enrollment_date` | Source record | Quarantine |
| `discontinuation_before_enrollment` | Temporal Consistency | `discontinuation_date`, `enrollment_date` | `discontinuation_date ≥ enrollment_date` | Source record | Quarantine |
| `discontinuation_before_randomization` | Temporal Consistency | `discontinuation_date`, `randomization_date` | `discontinuation_date ≥ randomization_date` | Source record | Quarantine |
| `invalid_subject_status` | Domain Validation | `subject_status` | Must be one of **SCREENING, ENROLLED, SCREEN_FAILED, DISCONTINUED, COMPLETED** | Controlled business values | Quarantine |
| `enrolled_without_arm` | Conditional Business Rule | `subject_status`, `arm_code` | Subjects with status **ENROLLED, DISCONTINUED, or COMPLETED** must have an `arm_code` | Source record | Quarantine |
| `unknown_arm_code` | Referential Integrity | `study_id`, `arm_code` | A non-null `arm_code` must be valid for the specified study | `dim_study_arm` | Quarantine |

### Subject Reference Dependencies

The Subject pipeline does not validate records in isolation. Several DQ controls depend on trusted reference and dimension datasets.

| Reference / Dimension | Used For |
|---|---|
| `dim_study` | Verify that the incoming study exists |
| `dim_site` | Verify site existence and site-to-study relationship |
| `dim_study_arm` | Verify that the treatment arm is valid for the study |
| `ref_sex` | Standardize raw sex values into controlled values |
| `ref_diagnosis` | Validate and enrich baseline diagnosis codes |
| `_source_file_name` | Derive and validate the logical source sequence date |